# Repaired original RoBERTa experiment

This tutorial calls the runnable code in `src/toxic_comments/neural.py`.
The original architecture uses **randomly initialized encoder weights**:
hidden size 512, 8 layers, 8 attention heads, feed-forward size 2,048,
512 position embeddings, and 256-token inputs. Only the real-data tokenizer
comes from `roberta-base`; pretrained encoder weights are never loaded.

Running this notebook with its defaults performs a tiny, offline engineering
check. It does **not** train the original architecture on the uploaded dataset
or produce a new real-data neural benchmark. Real training and test evaluation
are separate, explicit switches below. No raw comments or example predictions
are displayed.

## Environment

Use Python 3.12+ and install `requirements-neural.txt` and
`requirements-notebooks.txt` in the same environment as the notebook kernel.
Choose an official PyTorch build for your hardware; the repository's documented
CPU check uses PyTorch 2.8.0 and Transformers 4.57.1. The CLI dependency check is:

```bash
python scripts/check_neural_environment.py
```

Install dependencies before opening the notebook. There are no package-install
shell commands inside the computational cells.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (candidate for candidate in candidates
     if (candidate / "src" / "toxic_comments" / "neural.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))

from toxic_comments.data import LABELS
from toxic_comments.neural import (
    TrainConfig, check_environment, resource_diagnostic,
    run_smoke, run_development, evaluate_run,
)

print("Label order:", ", ".join(LABELS))

In [ ]:
environment = check_environment()
environment

## Resource diagnostic

The diagnostic constructs the original architecture on PyTorch's `meta` device
to count parameters. It does not allocate its full weights, download a tokenizer,
or run full-model training. The float32 parameter/gradient/Adam-moment memory
figure is an analytical lower bound. Activations and temporary tensors require
additional memory; peak memory and training time have not been measured.

A CPU-only forward/backward smoke check establishes mechanics, not whether a
full five-epoch run on roughly 160,000 comments is practical on this machine.

In [ ]:
resources = resource_diagnostic()
{
    key: resources[key]
    for key in (
        "parameter_count", "float32_parameter_memory_mib",
        "float32_parameters_gradients_adam_moments_lower_bound_mib",
        "cuda_available", "full_model_training_executed", "memory_note",
    )
}

## Tiny offline round trip

This check uses a one-layer model with hidden size 32 and a locally constructed,
train-only synthetic tokenizer. It performs real CPU forward/backward updates,
selects a checkpoint, restores and exports it, reloads model/tokenizer files,
then computes fresh validation predictions and freezes validation thresholds.
It evaluates the synthetic internal test partition once after freezing.

Temporary files are removed when the check finishes. Only engineering status
and assertions are displayed; synthetic metrics are not a toxicity benchmark.

In [ ]:
RUN_OFFLINE_SMOKE = True

if RUN_OFFLINE_SMOKE:
    with TemporaryDirectory(prefix="toxic-notebook-smoke-") as temporary:
        smoke = run_smoke(Path(temporary) / "run")
        smoke_status = {key: value for key, value in smoke.items() if key != "run_dir"}
    smoke_status
else:
    smoke_status = {"status": "skipped", "purpose": "optional offline engineering check"}

smoke_status

## Optional real-data development

Real training is disabled by default. Supply a private, labeled CSV containing
`comment_text` and all six binary labels. `severe_tox` is accepted as an alias
for `severe_toxic`. The uploaded `test_preprocessed.csv` has no labels and is
not a scoreable test dataset; do not pass it here.

The configuration below makes the supplied cleaned training CSV's cleaning
choices explicit: drop normalized-empty comments and keep conflicting duplicate
annotations in the same group. The loader's API/CLI defaults remain strict
(`error`) for both policies. Record and inspect the resulting data summary.

Normalized exact-text groups are split 70/15/15 with a fixed seed; this is not a
stratified split and does not detect paraphrases. The historical cohort has
already been inspected, so the held-out partition is an **internal exploratory
test**, not a pristine external benchmark.

Default microbatch 8 × 32 accumulation steps gives a nominal batch of 256.
Each accumulation group divides by its actual sample count, including the
short final group. Training positive weights use only training labels.
AMP is enabled only on CUDA; CPU always uses full precision. Gradient clipping
and scheduler updates follow actual optimizer updates. Workers default to 0;
seeds and deterministic algorithms are recorded without promising equality
across software versions or hardware.

`device="cuda"` below requests a GPU explicitly and fails clearly when CUDA is
unavailable. Select CPU only after considering the full architecture's cost.
`roberta-base` may download **tokenizer files only** when development is enabled;
provide a local tokenizer directory and `local_files_only=True` for an offline run.

In [ ]:
DATA_PATH = ROOT / "data" / "train_preprocessed.csv"
DEVELOPMENT_RUN = ROOT / "runs" / "roberta-original"
RUN_REAL_DEVELOPMENT = False
RUN_FROZEN_TEST_EVALUATION = False

TRAINING = TrainConfig(
    seed=42, epochs=5, patience=3,
    batch_size=8, accumulation_steps=32,
    learning_rate=4e-5, weight_decay=1e-2,
    warmup_fraction=0.1, max_length=256,
    max_gradient_norm=1.0,
    num_workers=0, cpu_threads=4,
    device="cuda", amp=True,
    tokenizer_name_or_path="roberta-base",
    local_files_only=False,
    conflict_policy="keep", empty_text_policy="drop",
)

In [ ]:
if RUN_REAL_DEVELOPMENT:
    if not DATA_PATH.is_file():
        raise FileNotFoundError("Set DATA_PATH to your private labeled training CSV.")
    development = run_development(DATA_PATH, DEVELOPMENT_RUN, config=TRAINING)
    print(json.dumps(development, indent=2))
else:
    print("Real-data neural development skipped; RUN_REAL_DEVELOPMENT is False.")

## Frozen export and fresh validation report

Development selects the lowest **sample-weighted validation BCE loss**. It
restores that checkpoint **before** exporting, verifies the saved tensors,
reloads the exported model and tokenizer, and performs new validation inference.
Thresholds and the validation report come from those new predictions, never
from last-epoch cached arrays. Label IDs and thresholds use the fixed order
printed above. A threshold-0.5 report is retained alongside thresholds selected
by validation F1.

The run manifest binds the source bytes, split indexes, configuration, source
code, model/tokenizer files, checkpoint, thresholds and reports by checksums.
These detect accidental changes; they are not a tamper-proof security scheme.
Runs, source snapshots, predictions and weights remain in ignored private
storage. The checkpoint supports best-model selection, not resumable training.

In [ ]:
if RUN_REAL_DEVELOPMENT:
    history = json.loads((DEVELOPMENT_RUN / "training_history.json").read_text())
    thresholds = json.loads((DEVELOPMENT_RUN / "thresholds.json").read_text())
    print(json.dumps({
        "best_checkpoint": history["best_checkpoint"],
        "fresh_validation_loss": history["fresh_validation_loss"],
        "thresholds": thresholds,
    }, indent=2))
else:
    print("No real-data neural result claimed by this notebook's default execution.")

## Optional one-time test evaluation

Enable this only for an already developed, frozen run. The model and thresholds
cannot be changed after seeing test results. The run claims its single evaluation
attempt before test inference; a second attempt is rejected, including after
an interrupted attempt. The CLI has no reset override. Public results contain
aggregate metrics; per-row arrays stay private.

Weighted-BCE sigmoid outputs are scores, not established calibrated probabilities.
Average precision, ROC-AUC and F1 have distinct meanings. Missing-class metrics
are explicitly undefined and carry coverage information. Real neural results
can be reported only after the optional real run actually completes.

In [ ]:
if RUN_FROZEN_TEST_EVALUATION:
    test_results = evaluate_run(DEVELOPMENT_RUN, device=TRAINING.device)
    print(json.dumps(test_results, indent=2))
else:
    print("Real-data neural test evaluation skipped; RUN_FROZEN_TEST_EVALUATION is False.")

## Equivalent CLI commands

Run from the repository root with the neural environment active:

```bash
PYTHONPATH=src python -m toxic_comments.neural check-environment
PYTHONPATH=src python -m toxic_comments.neural diagnose-resources
PYTHONPATH=src python -m toxic_comments.neural smoke --run-dir runs/neural-smoke
```

These optional commands explicitly start real development and score its frozen
internal exploratory test partition once:

```bash
PYTHONPATH=src python -m toxic_comments.neural develop --data data/train_preprocessed.csv --run-dir runs/roberta-original --device cuda --conflict-policy keep --empty-text-policy drop
PYTHONPATH=src python -m toxic_comments.neural evaluate --run-dir runs/roberta-original --device cuda
```

Each run directory must be new. Historical epoch tables in
`01_results_review.ipynb` remain historical saved evidence. The runnable neural
code and this engineering check do not reproduce those historical scores.